# Topic: L1 vs L2 Regularization (Lasso vs Ridge)

## Definition (30-second explanation)
L1 (Lasso) and L2 (Ridge) are regularization techniques that prevent machine learning models from overfitting by adding a penalty to the loss function. L1 penalizes the absolute value of weights, often shrinking them to exactly zero, while L2 penalizes the squared magnitude of weights, smoothly shrinking them towards zero.

## Why Interviewers Ask This
Interviewers want to see if you understand the bias-variance tradeoff and can correctly choose model constraints based on the dataset properties (like high dimensionality or multicollinearity).

## Core Concepts
*   **Penalty Terms:** L1 adds the sum of absolute weights ($\sum |w_i|$); L2 adds the sum of squared weights ($\sum w_i^2$).
*   **Geometry:** L1 creates a diamond-shaped constraint region; L2 creates a circular constraint region.
*   **Sparsity:** L1 creates a sparse model by setting coefficients to zero. L2 creates a dense model, keeping all features.

## When to Use
*   **Use L1 (Lasso):** When you have high-dimensional data, suspect many features are irrelevant, need automatic feature selection, or require a highly interpretable model.
*   **Use L2 (Ridge):** When you believe most features are useful, you have highly correlated features, or you want smooth and stable coefficient shrinkage for better prediction performance.

## Advantages
*   **L1:** Acts as an automatic feature selector, producing simpler, more interpretable models.
*   **L2:** Handles multicollinearity well and offers more stable weights when features are highly correlated.

## Limitations
*   **L1:** Sensitive and less stable with correlated features; it tends to randomly pick one correlated feature and set the others to zero.
*   **L2:** Does not perform feature selection (never sets coefficients to exactly zero), which can be an issue if model interpretability is the primary goal.

## Common Comparisons
*   **Feature Selection:** L1 = YES (automatic) | L2 = NO (keeps all)
*   **Stability with Correlation:** L1 = Less stable | L2 = More stable
*   **Effect on Weights:** L1 = Sets to zero | L2 = Shrinks toward zero

## Common Interview Traps
*   **Forgetting to scale:** Always scale features (e.g., `StandardScaler`) before applying regularization, otherwise features with larger scales will be penalized unfairly.
*   **Misunderstanding L1 with correlation:** Forgetting that L1 randomly drops correlated features, losing potentially useful information.
*   **Misinterpreting Alpha:** Forgetting that in `sklearn`, a higher `alpha` means *more* regularization.
*   **Assuming L2 is always better:** L1 is strictly better when a sparse model or feature selection is required.

## Python / SQL Syntax
```python
from sklearn.linear_model import Lasso, Ridge
from sklearn.preprocessing import StandardScaler

# Always scale features first
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# L1 Regularization
lasso = Lasso(alpha=0.5)
lasso.fit(X_train_scaled, y_train)

# L2 Regularization
ridge = Ridge(alpha=0.5)
ridge.fit(X_train_scaled, y_train)
```

## Important Formula
*   **L1 Loss:** $MSE + \alpha \sum |w_i|$
*   **L2 Loss:** $MSE + \alpha \sum w_i^2$
*   **Elastic Net:** Combines both constraints: $MSE + \alpha \cdot l1\_ratio \cdot \sum |w_i| + \alpha \cdot (1 - l1\_ratio) \cdot \sum w_i^2$

## 45-Second Interview Answer
Both L1 and L2 regularization prevent overfitting by penalizing large model weights. L1, or Lasso, penalizes the absolute value of the weights, which creates a diamond-shaped constraint region and drives many coefficients to exactly zero. This makes it perfect for feature selection and interpretability. L2, or Ridge, penalizes the squared weights, creating a circular constraint that smoothly shrinks all coefficients toward zero. L2 is typically better for datasets where most features are useful and handles multicollinearity much better than L1. If I need the best of both worlds, I use Elastic Net.

## Practice Questions:

### Q1: Business Case & Correlated Features

**Question:** 
You are building a logistic regression model for a bank to predict loan defaults. You have 300 features, but many of them are highly correlated (e.g., 'annual income', 'tax bracket'). The legal team requires the model to be strictly interpretable. Between L1 and L2, which would you choose to satisfy the legal team? What is the predictive risk of making that choice given the correlated features, and how might you address that risk?

**Answer:**
I would choose L1 (Lasso) regularization. The legal team requires strict interpretability, and L1 acts as an automatic feature selector by driving the coefficients of irrelevant features to exactly zero. L2 only shrinks them, leaving us with 300 non-zero coefficients which is too complex to explain to a customer.

The primary risk is how L1 handles multicollinearity: when features are highly correlated, L1 tends to randomly select one feature and push the rest to zero. This means it might drop an easily explainable feature like 'annual income' and keep a less interpretable proxy feature. 

To mitigate this, I would either manually drop the less interpretable correlated features during EDA prior to modeling, or I would use Elastic Net, which combines L1's feature selection with L2's ability to smoothly handle and group correlated features together.

**Interview Tips:**
*   Always tie mathematical properties (sparsity) to the business constraint (legal needs interpretability).
*   If you identify a model's weakness (random dropping of correlated features), always proactively offer a solution (Elastic Net or manual EDA).

### Q2: 
You are given a Pandas DataFrame named credit_df.

Write the Python (Sklearn/Pandas) code to train an L1 regularized logistic regression model on this dataset. You can assume the data is already clean and has no missing values.

In [6]:
# Data:
import pandas as pd

# Generate mock data
data = {
    'age': [25, 45, 60, 32, 50],
    'credit_score': [600, 750, 800, 550, 720],
    'annual_income': [45000.0, 120000.0, 95000.0, 38000.0, 110000.0],
    'default': [1, 0, 0, 1, 0]  # Target variable
}

credit_df = pd.DataFrame(data)

In [7]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

X = credit_df.drop('default', axis=1)
y = credit_df['default']

# Define range of C values (inverse of regularization strength)
c_values = [0.001, 0.01, 0.5, 1]

# Use a pipeline to prevent data leakage during CV
pipeline = Pipeline(steps=[
    ('preprocessor', StandardScaler()),
    # solver='liblinear' is required for L1 penalty in Logistic Regression
    ('model', LogisticRegressionCV(Cs=c_values, penalty='l1', cv=3, random_state=42, solver='liblinear'))
])

pipeline.fit(X, y)

print(f"Best C Value: {pipeline.named_steps['model'].C_}")

Best C Value: [1.]


/home/shail/interview-prep/interview_env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:2092: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/home/shail/interview-prep/interview_env/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:2123: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' and 'Cs' instead. Use l1_ratios=(0,) instead of penalty='l2', l1_ratios=(1,) instead of penalty='l1', l1_ratios set to floats between 0 and 1 instead of penalty='elasticnet', and Cs=(np.inf,) instead of penalty=None.
  warnings.warn(
/home/shail/interview-prep/interview_env/lib/python3.12/site-packages/sklearn/linear_model/

**Interview Tips:**
*   **The Solver Trap:** Default `LogisticRegression` uses `solver='lbfgs'`, which does not support L1. You MUST specify `solver='liblinear'` or `solver='saga'`.
*   **The Scaling Trap:** Always scale features before applying L1/L2, ideally inside a `Pipeline` to prevent data leakage during cross-validation.
*   **C vs Alpha:** Remember that for `LogisticRegression`, the parameter is `C` (Inverse of regularization strength). Smaller `C` = More regularization. For `Lasso/Ridge`, the parameter is `alpha`. Larger `alpha` = More regularization.

### Q3:
**Lasso, Ridge, and ElasticNet Comparison**

You are tasked with predicting house_price using a regression model. You want to train and evaluate L1 (Lasso), L2 (Ridge), and ElasticNet to compare their behavior.

**Your Task:**
Write the Python code to:

- Scale the features.

- Train a Lasso, Ridge, and ElasticNet model (using alpha=0.5 for all, and l1_ratio=0.5 for ElasticNet).

- Print the number of coefficients that were driven to exactly zero by the Lasso model.

In [10]:
# Data:
import pandas as pd
from sklearn.datasets import make_regression

# Generate mock regression data
X_mock, y_mock = make_regression(n_samples=100, n_features=10, n_informative=5, noise=15, random_state=42)

# Create DataFrame
columns = [f'feature_{i}' for i in range(1, 11)]
house_df = pd.DataFrame(X_mock, columns=columns)
house_df['house_price'] = y_mock

# Your features and target
X = house_df.drop('house_price', axis=1)
y = house_df['house_price']

In [11]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'Ridge': Ridge(alpha=0.5),
    'Lasso': Lasso(alpha=0.5),
    'ElasticNet': ElasticNet(alpha=0.5, l1_ratio=0.5)
}

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('scaler', StandardScaler()),
        ('model', model)
    ])
    
    pipeline.fit(X_train, y_train)
    
    # Extract coefficients from the pipeline
    coefs = pipeline.named_steps['model'].coef_
    
    # Count how many coefficients were driven to exactly zero
    zero_coefs = np.sum(coefs == 0)
    print(f"{name} - Number of features dropped (zero coefficients): {zero_coefs}")

Ridge - Number of features dropped (zero coefficients): 0
Lasso - Number of features dropped (zero coefficients): 1
ElasticNet - Number of features dropped (zero coefficients): 0


**Interview Tips:**
*   **Pipeline Attribute Extraction:** To inspect a model's attributes (like `.coef_` or `.feature_importances_`) when it's wrapped in a `Pipeline`, you must use `pipeline.named_steps['step_name'].attribute`. 
*   **Lasso Verification:** Interviewers often ask you to prove Lasso performed feature selection by explicitly counting the `0` coefficients using `np.sum(coefs == 0)`.